In [ ]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import warnings
import torch

warnings.filterwarnings("ignore", category=UserWarning)  # cleaner output



MODEL_NAME = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
EMBEDDINGS_PATH = 'ae_embeddings_sapbert.npy'
UNIQUE_AES_PATH = 'unique_aes.txt'          
CLUSTERS_OUTPUT = 'ae_clusters.csv'
SIM_MATRIX_PATH = 'ae_similarity_matrix.npy'   
TOP_K = 20                                    
N_CLUSTERS = None                              
DISTANCE_THRESHOLD = 0.8                       
BATCH_SIZE = 64
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'


if os.path.exists(UNIQUE_AES_PATH):
    if UNIQUE_AES_PATH.endswith('.npy'):
        unique_aes = np.load(UNIQUE_AES_PATH, allow_pickle=True).tolist()
    else:
        with open(UNIQUE_AES_PATH, 'r', encoding='utf-8') as f:
            unique_aes = [line.strip() for line in f if line.strip()]
else:
    raise FileNotFoundError(f"Could not find {UNIQUE_AES_PATH}")


model = SentenceTransformer(MODEL_NAME)


if not os.path.exists(EMBEDDINGS_PATH):
    ae_embeddings = model.encode(
        unique_aes,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        device=DEVICE
    )
    np.save(EMBEDDINGS_PATH, ae_embeddings)
else:
    ae_embeddings = np.load(EMBEDDINGS_PATH)


test_terms = [
    'acute myocardial infarction',
    'cardiac arrest',
    'heart failure',
    'stroke',
    'headache',
    'diarrhoea',
    'rash',
    'depression',
    'anxiety'
]

test_embeds = model.encode(test_terms, normalize_embeddings=True, batch_size=32)
sim_matrix_test = cosine_similarity(test_embeds)
mi_idx = test_terms.index('acute myocardial infarction')
mi_sims = sim_matrix_test[mi_idx]

for term, score in sorted(
    zip(test_terms, mi_sims),
    key=lambda x: x[1],
    reverse=True
):
    if term != 'acute myocardial infarction':
        print(f"{score:>6.4f} | acute myocardial infarction vs {term.upper()}")



# if not os.path.exists(SIM_MATRIX_PATH):
#     print("\nComputing full cosine similarity matrix...")
#     sim_matrix = cosine_similarity(ae_embeddings)
#     np.save(SIM_MATRIX_PATH, sim_matrix)
#     print(f"→ Full similarity matrix saved ({sim_matrix.shape})")
# else:
#     sim_matrix = np.load(SIM_MATRIX_PATH)
#     print("→ Full similarity matrix loaded")

from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=TOP_K+1, metric='cosine', algorithm='auto')
nn.fit(ae_embeddings)

distances, indices = nn.kneighbors(ae_embeddings)


top_k_similar = {}
for i, ae in enumerate(unique_aes):
    sim_scores = 1 - distances[i]  
    similar_aes = [unique_aes[j] for j in indices[i][1:]] 
    top_k_similar[ae] = list(zip(similar_aes, sim_scores[1:]))

if 'acute myocardial infarction' in top_k_similar:
    
    for term, sim in top_k_similar['acute myocardial infarction'][:5]:
        print(f"  {sim:.4f}  {term}")



Z = linkage(ae_embeddings, method='ward', metric='euclidean')

plt.figure(figsize=(14, 7))
dendrogram(Z, no_labels=True, truncate_mode='level', p=5)
plt.title('Hierarchical Clustering Dendrogram (Ward linkage)')
plt.xlabel('AE terms (truncated)')
plt.ylabel('Distance')
plt.savefig('ae_dendrogram.png', dpi=150, bbox_inches='tight')
plt.close()

clustering = AgglomerativeClustering(
    n_clusters=N_CLUSTERS,
    distance_threshold=DISTANCE_THRESHOLD,
    linkage='ward',
    metric='euclidean'
)

labels = clustering.fit_predict(ae_embeddings)

n_clusters_found = len(set(labels)) - (1 if -1 in labels else 0)
print(f"→ Found {n_clusters_found} clusters (threshold = {DISTANCE_THRESHOLD})")


cluster_df = pd.DataFrame({
    'ae_term': unique_aes,
    'cluster_id': labels
})

cluster_df.to_csv(CLUSTERS_OUTPUT, index=False)

cluster_sizes = cluster_df['cluster_id'].value_counts().head(10)
print("\nLargest clusters:")
print(cluster_sizes)

if 'acute myocardial infarction' in cluster_df['ae_term'].values:
    mi_cluster = cluster_df[cluster_df['ae_term'] == 'acute myocardial infarction']['cluster_id'].iloc[0]
    cluster_members = cluster_df[cluster_df['cluster_id'] == mi_cluster]['ae_term'].tolist()
   

In [2]:
model = SentenceTransformer('cambridgeltl/SapBERT-from-PubMedBERT-fulltext')

No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

def DE_matrix(df_pair):
    drug_event = (df_pair.groupby(['event','drug'])).size().unstack(fill_value=0)
    return drug_event
def co_occurence(de_mat):
    X= normalize(de_mat.values,norm='12')
    simi =cosine_similarity(X)
    return pd.DataFrame(simi,index=de_mat.index,columns=de_mat.index)

def hybrid_simi(eve_i , eve_j , sementic_simi,cooc_simi,alpha=.5):
    sem = sementic_simi.loc[eve_i,eve_j]
    cooc = cooc_simi.loc[eve_j,eve_j]
    return alpha*sem+(1-alpha)*cooc

def hybrid_neighbours(tar_eve , sementic_simi , cooc_simi,top_k,alpha =.5):
    scores =[]
    for  e in sementic_simi.columns:
        if e==tar_eve:
            continue
        score = hybrid_simi(tar_eve,e,sementic_simi,cooc_simi,alpha)
    scores.sort(key =lambda x:x[1],reverse=True)
    return scores[:top_k]
def pooling_counts(drg,eve,nbgr,combo_cnt , weight=1):
    a = combo_cnt.get((drg,eve),0)
    for ev,sim in nbgr:
        w = min(sim,weight)
        a += w*combo_cnt.get((drg,ev),0)
    return a